# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-12\data\lev-12_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('ITEM_CODE', String),
        ('QUANTITY', Float64),
        ('VALUE', Int64),
        ('MULTIPLIER', Int64)])

# Useful Variables

In [5]:
cols = [
'ITEM_CODE',
'QUANTITY',
'VALUE',
'MULTIPLIER',
]


In [6]:
df = pdf.select(cols)

In [7]:
df.head(2).collect()

ITEM_CODE,QUANTITY,VALUE,MULTIPLIER
str,f64,i64,i64
"""031""",4.0,2200,16669
"""034""",1.0,1200,16669


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

ITEM_CODE,QUANTITY,VALUE,MULTIPLIER
u32,u32,u32,u32
45,117,22789,23565


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_368\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
ITEM_CODE,9414702.0,0.0,345.93536,93.261333,30.0,360.0,370.0,382.0,399.0
QUANTITY,7713350.0,1701352.0,3.977238,4.142578,0.0,2.0,3.0,5.0,200.0
VALUE,9414702.0,0.0,1971.865046,3842.482383,1.0,500.0,900.0,1800.0,204550.0
MULTIPLIER,9414702.0,0.0,110362.092951,78192.153619,369.0,55037.0,113283.0,149939.0,2366902.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))